# Sense of Cellf — Label-Free Cell Re-ID Pipeline

**Dataset:** DIC-C2DH-HeLa (Cell Tracking Challenge)  
**Method:** Contrastive fine-tuning of Cell-DINO (DINOv2 on microscopy), no GT labels during training  
**Eval:** CMC @ rank-1/5/10 + silhouette score on sequence 02

---
**Before running:** Set *Runtime → Change runtime type → T4 GPU*

## 1 · Install dependencies

In [ ]:
%%capture
!pip install tifffile cellpose scikit-image transformers scikit-learn torchvision
# cellpose pulls in torch; transformers gives us DINOv2 via AutoModel
print('Done')

## 2 · Mount Google Drive and set paths

Upload the DIC-C2DH-HeLa dataset to your Drive, then adjust `DATA_ROOT` below.

Expected layout:
```
MyDrive/DIC-C2DH-HeLa/
  01/t000.tif ... t083.tif
  01_GT/TRA/man_track.txt
  01_GT/TRA/man_track000.tif ...
  02/t000.tif ... t083.tif
  02_GT/TRA/man_track.txt
  02_GT/TRA/man_track000.tif ...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DATA_ROOT    = Path('/content/drive/MyDrive/DIC-C2DH-HeLa')  # <-- adjust if needed
CHECKPOINT_DIR = Path('/content/checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

SEQ01_DIR    = DATA_ROOT / '01'
SEQ01_GT_DIR = DATA_ROOT / '01_GT' / 'TRA'
SEQ02_DIR    = DATA_ROOT / '02'
SEQ02_GT_DIR = DATA_ROOT / '02_GT' / 'TRA'

# Quick sanity check
for p in [SEQ01_DIR, SEQ01_GT_DIR, SEQ02_DIR, SEQ02_GT_DIR]:
    status = '✓' if p.exists() else '✗ NOT FOUND'
    print(f'  {status}  {p}')

## 3 · Clone the pipeline code from GitHub

If the repo is private, replace with your clone URL or upload the `.py` files manually to `/content/sense-of-cellf/`.

In [ ]:
import os

REPO_DIR = Path('/content/sense-of-cellf')

if not REPO_DIR.exists():
    !git clone https://github.com/merepixel/sense-of-cellf.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# Add repo to Python path so modules import correctly
import sys
sys.path.insert(0, str(REPO_DIR))
print('Repo ready at', REPO_DIR)

# Force Python to reload cached modules after a git pull
import importlib
for mod_name in ['data_loader', 'detector', 'embedder', 'train', 'evaluate']:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])
        print(f'  reloaded {mod_name}')


## 4 · Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

USE_GPU = torch.cuda.is_available()

## 5 · Load sequence data

Preview frames and GT annotation stats.

In [ ]:
import matplotlib.pyplot as plt
from data_loader import SequenceData

seq01 = SequenceData(SEQ01_DIR, gt_tra_dir=SEQ01_GT_DIR)
print(f'Sequence 01: {seq01.num_frames()} frames')
print(f'Tracklets in man_track.txt: {len(seq01.tracklets)}')

# Show first frame
fidx, frame = seq01.frames[0]
plt.figure(figsize=(6, 6))
plt.imshow(frame[:, :, 0], cmap='gray')
plt.title(f'Sequence 01 — frame {fidx}')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Tracklet length distribution
lengths = [(e - b + 1) for b, e, _ in seq01.tracklets.values()]
print(f'Tracklet lengths: min={min(lengths)}, max={max(lengths)}, median={sorted(lengths)[len(lengths)//2]}')
print(f'Tracklets ≥ 10 frames: {sum(l >= 10 for l in lengths)}')

divs = seq01.get_division_frames()
print(f'Division events: {len(divs)}')

## 6 · Run CellPose on a sample frame

Verify segmentation looks reasonable before training.

In [ ]:
import numpy as np
from detector import CellDetector

detector = CellDetector(crop_size=96, use_gpu=USE_GPU)

# Detect on frame 0
sample_fidx, sample_frame = seq01.frames[5]   # frame 5 for variety
sample_cells = detector.detect_frame(sample_fidx, sample_frame)
print(f'Detected {len(sample_cells)} cells in frame {sample_fidx}')

# Visualise bounding boxes
import matplotlib.patches as patches
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(sample_frame[:, :, 0], cmap='gray')
for c in sample_cells:
    r0, c0, r1, c1 = c.bbox
    rect = patches.Rectangle((c0, r0), c1 - c0, r1 - r0,
                               linewidth=1, edgecolor='cyan', facecolor='none')
    ax.add_patch(rect)
ax.set_title(f'Frame {sample_fidx}: {len(sample_cells)} cells detected')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Preview first 8 crops
n_show = min(8, len(sample_cells))
fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2))
for i, ax in enumerate(axes):
    ax.imshow(sample_cells[i].crop[:, :, 0], cmap='gray')
    ax.axis('off')
    ax.set_title(f'#{i}', fontsize=8)
plt.suptitle('Sample crops (96×96)', y=1.02)
plt.tight_layout()
plt.show()

## 7 · Load Cell-DINO embedder

Downloads `recursionpharma/OpenPhenom` from HuggingFace (~350 MB, cached after first run).

In [ ]:
from embedder import CellDINOEmbedder

embedder = CellDINOEmbedder(device='cuda' if USE_GPU else 'cpu')
print(f'Embedding dimension: {embedder.embed_dim}')
print(f'Device: {embedder.device}')

In [ ]:
# Quick embedding test
import torch
test_crops = [c.crop for c in sample_cells[:4]]
with torch.no_grad():
    test_embs = embedder.embed_crops(test_crops, no_grad=True)
print('Embeddings shape:', test_embs.shape)   # should be (4, 768)
print('L2 norms:', test_embs.norm(dim=-1).cpu().numpy())  # should be ~1.0

## 8 · Training

Fine-tunes Cell-DINO on sequence 01 with contrastive learning — **no GT labels used**.

- Positive pairs: spatial matching (centroid distance + IoU) across consecutive frames
- Hard negatives: top-k cosine-similar cells within the same frame
- Loss: NT-Xent (InfoNCE)
- Checkpoint selection: silhouette score on held-out frames

In [ ]:
# Training config — adjust as needed
TRAIN_CONFIG = dict(
    seq_dir        = SEQ01_DIR,
    output_dir     = CHECKPOINT_DIR,
    epochs         = 20,
    lr             = 1e-5,
    weight_decay   = 1e-4,
    temperature    = 0.07,
    hard_neg_k     = 4,
    silhouette_every = 2,
    held_out_frac  = 0.2,
    max_dist_px    = 50.0,
    iou_threshold  = 0.2,
    crop_size      = 96,
    use_gpu        = USE_GPU,
    seed           = 42,
)
print('Training config:')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
from train import train
train(**TRAIN_CONFIG)

## 9 · Evaluation

Evaluates on sequence 02 using GT labels for the first time.  
Reports CMC @ rank-1/5/10 and silhouette score for:
- **Frozen baseline** — Cell-DINO with no fine-tuning
- **Fine-tuned** — best checkpoint from training

In [ ]:
from evaluate import evaluate

BEST_CKPT = CHECKPOINT_DIR / 'best_checkpoint.pt'

evaluate(
    seq_dir     = SEQ02_DIR,
    gt_tra_dir  = SEQ02_GT_DIR,
    checkpoint  = BEST_CKPT if BEST_CKPT.exists() else None,
    crop_size   = 96,
    use_gpu     = USE_GPU,
    ranks       = (1, 5, 10),
)

## 10 · Save checkpoints to Drive (optional)

Colab VMs are ephemeral — copy checkpoints to Drive to persist them.

In [ ]:
import shutil

DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/sense-of-cellf-checkpoints')
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

for ckpt in CHECKPOINT_DIR.glob('*.pt'):
    dest = DRIVE_CKPT_DIR / ckpt.name
    shutil.copy2(ckpt, dest)
    print(f'Copied {ckpt.name} → {dest}')